## Atlas basic usage

This notebook steps through the easiest way to run Atlas. The demo will create and run an analysis with a fixed-slope powerlaw CURN and intrinsic pulsar red noise. Not included in this demo: varied white noise, varied timing model, CWs.

In [1]:
import sys
## placing Atlas on the path
sys.path.append('../')

In [2]:
## imports
import glob
import numpy as np
import pickle
import json
import os
import matplotlib.pyplot as plt
from corner import corner
from ATLAS.data import PTA_Data
from ATLAS.model_builder import ModelBuilder
from ATLAS.psd_functions import powerlaw, free_spectrum, hd_orf, gt_orf, bin_orf
from functools import partial
import jax.numpy as jnp
import jax.random as jrandom
from ATLAS.samplers.canetoadracing import model_maker
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value
numpyro.distributions.distribution = sys.modules['numpyro.distributions.distribution']

JAX 64-bit mode automatically enabled successfully.


/home/awc/.local/share/mamba/envs/atlas-env-1/lib/python3.12/site-packages/tqdm_joblib/__init__.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
## if tempo2 path is giving you trouble:
os.environ['TEMPO2'] = '/home/awc/.local/share/mamba/envs/atlas-env-1/share/tempo2'

## Load the dataset

Two options: NG15 and the IPTA MDC-1. Warning: NG15 requires a fair bit of memory to run on GPU

### NG15

In [4]:
## point to the datasets. We'll use NG15.
datapath = '../datasets/NG15/'
ng15 = True
mdc1 = False

In [5]:
## note that the pre-fit feather files only work if we are note varying the timing model
from enterprise.pulsar import FeatherPulsar
import glob

feather_files = glob.glob('../datasets/NG15/feathers/*.feather')
psrs = []
for feather_file in feather_files:
    psr = FeatherPulsar.read_feather(feather_file)
    psrs.append(psr)
## get rid of 1713
psrs = [psr for psr in psrs if psr.name != 'J1713+0747']
with open('../datasets/NG15/v1p1_wn_dict.json', 'r') as f:
    noise_dict = json.load(f)

## also load the par and tim files directly
pnames = ['B1855+09', 'B1937+21', 'B1953+29', 'J0023+0923', 'J0030+0451', 'J0340+4130', 'J0406+3039', 
    'J0437-4715', 'J0509+0856', 'J0557+1551', 'J0605+3757', 'J0610-2100', 'J0613-0200', 'J0636+5128', 
    'J0645+5158', 'J0709+0458', 'J0740+6620', 'J0931-1902', 'J1012+5307', 'J1012-4235', 'J1022+1001', 
    'J1024-0719', 'J1125+7819', 'J1312+0051', 'J1453+1902', 'J1455-3330', 'J1600-3053', 'J1614-2230', 
    'J1630+3734', 'J1640+2224', 'J1643-1224', 'J1705-1903', 
    'J1713+0747', 
    'J1719-1438', 'J1730-2304', 'J1738+0333', 'J1741+1351', 'J1744-1134', 'J1745+1017', 'J1747-4036', 
    'J1751-2857', 'J1802-2124', 'J1811-2405', 'J1832-0836', 'J1843-1113', 'J1853+1303', 'J1903+0327', 
    'J1909-3744', 'J1910+1256', 'J1911+1347', 'J1918-0642', 'J1923+2515', 'J1944+0907', 'J1946+3417', 
    'J2010-1323', 'J2017+0603', 'J2033+1734', 'J2043+1711', 'J2124-3358', 'J2145-0750', 'J2214+3000', 
    'J2229+2643', 'J2234+0611', 'J2234+0944', 'J2302+4442', 'J2317+1439', 'J2322+2057']

parfiles_ref = sorted(glob.glob('/*.par'))
timfiles_ref = sorted(glob.glob('/*.tim'))
parfiles = []
for pname in pnames:
    for p in parfiles_ref:
        if pname in p and not 'ao' in p and not 'gbt' in p:
            parfiles.append(p)
timfiles = []
for pname in pnames:
    for p in timfiles_ref:
        if pname in p and not 'ao' in p and not 'gbt' in p:
            timfiles.append(p)

/home/awc/.local/share/mamba/envs/atlas-env-1/lib/python3.12/site-packages/enterprise/pulsar.py:741: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  f = feather.read_table(filename)


FeatherPulsar.read_feather: cannot find fitpars in feather file ../datasets/NG15/feathers/v1p1_de440_pint_bipm2019-J1312+0051.feather.
FeatherPulsar.read_feather: cannot find setpars in feather file ../datasets/NG15/feathers/v1p1_de440_pint_bipm2019-J1312+0051.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file ../datasets/NG15/feathers/v1p1_de440_pint_bipm2019-J1312+0051.feather.
FeatherPulsar.read_feather: cannot find fitpars in feather file ../datasets/NG15/feathers/v1p1_de440_pint_bipm2019-J0023+0923.feather.
FeatherPulsar.read_feather: cannot find setpars in feather file ../datasets/NG15/feathers/v1p1_de440_pint_bipm2019-J0023+0923.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file ../datasets/NG15/feathers/v1p1_de440_pint_bipm2019-J0023+0923.feather.
FeatherPulsar.read_feather: cannot find fitpars in feather file ../datasets/NG15/feathers/v1p1_de440_pint_bipm2019-J1910+1256.feather.
FeatherPulsar.read_feather: cannot find setpars in feathe

### MDC-1

In [7]:
datapath = '../tests/fixtures/data/mdc1_36.npz'
ng15 = False
mdc1 = True

In [8]:
## load the stripped-down MDC pulsars
from tests.fixtures import pulsar
psrs, prov = pulsar.load_fixture(datapath)

## setup for the MDC pulsars
parfiles, timfiles, noise_dict = None, None, None

## Setting up the analysis

Set your save directory:

In [9]:
savedir = ...
savedir = '/home/awc/Documents/NANOGrav/global_fit/tests/demo_test_1/'
os.makedirs(savedir, exist_ok=True)

Now we can set up our data class. Here we tell Atlas everything it needs to build bases, etc..

In [10]:
help(PTA_Data)

Help on class PTA_Data in module ATLAS.data:

class PTA_Data(builtins.object)
 |  PTA_Data(psrs, adaptus_basis=None, num_gwb_bins=None, num_irn_bins=None, num_dm_bins=None, num_det_bins=None, adaptus_size=None, fixed_white_noise_params=None, linear_timing=False, marg_timing=False, diag_white_cov=False, fixed_res=False, timfiles=None, parfiles=None, noise_dict=None, dm_ref_freq=1400)
 |
 |  A class to hold static data attributes of the PTA dataset.
 |
 |  This class is intended to hold all the static data attributes of the PTA
 |  dataset, such as the pulsar objects, their TOAs, positions, and timespans.
 |  See the following attributes for details.
 |
 |  Attributes
 |  ----------
 |  psrs : list
 |      A list of enterprise-like pulsar objects.
 |  npsrs : int
 |      The number of pulsars in the dataset.
 |  psr_names : list
 |      A list of pulsar names corresponding to the pulsar objects.
 |  fixed_wn : bool
 |      A flag indicating whether the white noise matrices are fixed.
 | 

In [11]:
data = PTA_Data(psrs, 
                # adaptus_basis = [np.load(path_to_Adaptus_bases)[0]
                #                   for psr in psrs],
                num_gwb_bins = 14,
                num_irn_bins = 30,
                num_dm_bins = None,
                # adaptus_size = 500, 
                fixed_white_noise_params = None,
                linear_timing = True,
                marg_timing = False,
                diag_white_cov = False,
                fixed_res = False,
                timfiles = timfiles,
                parfiles = parfiles,
                noise_dict = noise_dict,
                dm_ref_freq = 1400
                )

In [12]:
## AC -- this doesn't seem to be necessary and doesn't work
# ## generate the design matrices
# Mmats = [np.load(path_to_Mmat)[0] for psr in psrs] #1-column Mmat (phase offset)
# data.add_timing_design_matrix(Mmats)

The ModelBuilder builds the model to pass to Numpyro.

In [13]:
help(ModelBuilder)

Help on class ModelBuilder in module ATLAS.model_builder:

class ModelBuilder(builtins.object)
 |  ModelBuilder(data, explicit_timing_model_params_to_sample=None)
 |
 |  Methods defined here:
 |
 |  __init__(self, data, explicit_timing_model_params_to_sample=None)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  make_red_noise(self, red_noise_combination_string, use_pulsar_tspan=False, irn_psd_function=None, gwb_psd_function=None, det_delay_function=None, orf_function=None, dm_psd_function=None, upper_bound_orf=None, lower_bound_orf=None, irn_lower_bound_psd=None, irn_upper_bound_psd=None, dm_lower_bound_psd=None, dm_upper_bound_psd=None, gwb_lower_bound_psd=None, gwb_upper_bound_psd=None, det_parameter_bounds=None, gt_psd_val=None)
 |
 |  make_timing_model(self, enterprise_data=True)
 |      Build the non-linear timing model.
 |
 |      The JUG import is deliberately function-level. It is the only thing in
 |      this module that needs JUG, and JUG is not

In [14]:
## the ModelBuilder class
constructor = ModelBuilder(data = data)

In [15]:
## set up the white noise
wn = constructor.make_white_noise(stabilize_TNT=True)
# theta0_wn = wn.prior_draw()
# theta0_wn = wn.params_dict_to_vector(noise_dict)
# theta0_wn[:5]

Constructing the white noise cov matrix for J2317+1439: 100%|████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:00<00:00, 338.24it/s]


In [16]:
## pull the white noise prior bounds for later
wn_lower_bound, wn_upper_bound = wn.get_prior_bounds()

Here we set up the red noise model. At present, we also pass a "setup" string here that tells Atlas how to share (or not share) bases matrices. The example here,

```ltm|unc+cor->unc```

Indicates we are

1. Using a linear timing model (```ltm|```)
2. Using both an uncorrelated signal (the intrinsic red noise, ```unc```) and
3. a correlated signal (the GWB, ```cor```), but
4. Atlas should use a single basis for both (the basis of the uncorrelated signal, hence ```->unc```)

In [17]:
rn = constructor.make_red_noise("ltm|unc+cor->unc",
                irn_psd_function = partial(free_spectrum),
                gwb_psd_function = partial(powerlaw),
                orf_function = hd_orf,
                irn_lower_bound_psd = jnp.ones(30) * -9, #jnp.array([-20, 0]),
                irn_upper_bound_psd = jnp.ones(30) * -2, #jnp.array([-11, 7]),
                gwb_lower_bound_psd = jnp.array([-18, 0]),
                gwb_upper_bound_psd = jnp.array([-11, 7])
                    )

In [18]:
## explicitly get the raw residuals
raw_res = jnp.concat(data.raw_residuals)

In [24]:
## non-varied white noise case (NG15):
if ng15:
    ## construct the helpers
    ## these are the cached matrices used to calculate TNT and TNr
    helpers = rn.get_helpers(reff = raw_res, 
                    white_noise_params = wn.params_dict_to_vector(data.noise_dict))
    vary_white_noise = False
elif mdc1:
    helpers = None
    ## fit the white noise simultaneously
    vary_white_noise = True

In [25]:
## define the Numpyro NUTS engine
nuts_kernel = NUTS(
                    model = model_maker, #############NOTE: check the model_maker to see what it does.
                    target_accept_prob = 0.8,
                    # init_strategy = init_to_value(values={"red_noise": x0, 
                    #                                      "z_a": jnp.zeros((data.npsrs, rn.nmodes))}),
                    max_tree_depth = 10)

In [26]:
## set up the sampler
mcmc = MCMC(
    sampler=nuts_kernel,
    num_warmup=1000,
    num_samples=5000,
    num_chains=1,
)


In [27]:
## run!
mcmc.run(jrandom.key(170817), 
                extra_fields=("~z.z_a",),
                raw_residuals = raw_res, 
                super_sig = rn,
                vary_white = vary_white_noise,
                wn_lower_bound = wn_lower_bound,
                wn_upper_bound = wn_upper_bound,
                tm_model = None,
                helpers = helpers,
                save_red_coeff = False,
                marg_over_non_gwb = False)

sample: 100%|███████████████████████████████████████████████████████████████████████████████████████| 510/510 [08:27<00:00,  1.00it/s, 255 steps of size 1.70e-02. acc. prob=0.82]


In [28]:
samples = mcmc.get_samples()

In [31]:
samples.keys()

dict_keys(['red_noise', 'white_noise'])

In [33]:
samples['white_noise'].shape

(500, 72)

In [34]:
corner(samples)
plt.savefig(savedir+'corner.png')

In [ ]:
np.savez_compressed(savedir + '/chain.npz', **samples)